In [2]:
pip install opencv-python

  Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_14_0_x86_64.whl.metadata (19 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_14_0_x86_64.whl (32.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 4.6 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [opencv-python]0m [opencv-python]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.62.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/admin/fl_env/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 6.3 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/admin/fl_env/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import cv2
import json
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [ ]:
def extract_frames(video_path, output_dir, label, max_frames=30):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return 0

    indices = set(sorted(random.sample(range(total), min(max_frames, total))))
    stem = Path(video_path).stem
    label_dir = os.path.join(output_dir, str(label))
    os.makedirs(label_dir, exist_ok=True)

    saved = 0
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx in indices:
            cv2.imwrite(os.path.join(label_dir, f"{stem}_f{frame_idx:05d}.jpg"), frame)
            saved += 1
        frame_idx += 1

    cap.release()
    return saved


def build_dataset(shoplifting_dir, normal_dir, output_dir, frames_per_video=30):
    total = 0
    for video_file in Path(shoplifting_dir).glob("*.mp4"):
        n = extract_frames(str(video_file), output_dir, label=1, max_frames=frames_per_video)
        total += n
        print(f"[shoplifting] {video_file.name}: {n} frames")

    for video_file in Path(normal_dir).glob("*.mp4"):
        n = extract_frames(str(video_file), output_dir, label=0, max_frames=frames_per_video)
        total += n
        print(f"[normal] {video_file.name}: {n} frames")

    print(f"\nTotal frames extracted: {total}")


# Update these paths to your UCF-Crime download location
SHOPLIFTING_DIR = "UCF_Crime/Shoplifting"
NORMAL_DIR = "UCF_Crime/Normal"
FRAMES_DIR = "data/frames"

build_dataset(SHOPLIFTING_DIR, NORMAL_DIR, FRAMES_DIR, frames_per_video=30)